# Periodogram-CNN Diagnostic

**Why this exists:** Two OOF runs of `cnn_periodogram.ipynb` collapsed to
R=1.000 / ROC=0.500 / threshold ~= 0.49 — the signature of a model outputting
`~0.5` for every star regardless of input. This notebook runs ONE fold for
20 epochs with **per-epoch loss + gradient norm + val PR-AUC + logit std**
traces to pinpoint the failure mode.

Three architecture variants in sequence:
1. **Original (Conv1d + BatchNorm, AdamW + OneCycleLR)** — the failing config.
2. **Conv1d + LayerNorm** — isolates whether BN with batch=32 on heavy-tailed
   log1p(power) is killing gradients.
3. **Plain MLP on flattened periodogram** — if MLP also fails, the input
   carries no extractable signal at all; if MLP succeeds, the conv/BN architecture
   is the problem.

~5 minutes total on Kaggle (1 fold, 20 epochs each, 3 variants).

## 1. Imports + config

In [ ]:
import os
os.environ['PYTHONHASHSEED'] = '0'
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from astropy.timeseries import LombScargle
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
import time

seed = 42
np.random.seed(seed); torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Same config as cnn_periodogram.ipynb (the failing notebook)
CHANNELS = ['rv']
N_FREQ = 500
LS_METHOD = 'fast'
N_REPS, N_FOLDS = 5, 10
N_EPOCHS_DIAG = 20   # shorter than OOF's 60 -- just enough to see the loss trajectory
BATCH = 32
LR = 3e-4
WD = 1e-4

OBS_PKL = '/kaggle/input/datasets/maanav0114/harps-n-dataset/observations.pkl'
observations = pd.read_pickle(OBS_PKL)
print(f'Stars: {observations["star_name"].nunique()}  obs: {len(observations)}')
print(f'device: {device}')

## 2. Star vocabulary + labels

In [ ]:
grouped = observations.groupby('star_name', sort=True)
stars = list(grouped.groups.keys())
labels = np.array([int(grouped.get_group(s)['has_exoplanets'].iloc[0]) for s in stars], dtype=int)
n_stars = len(stars)
print(f'n_stars={n_stars}, pos={int(labels.sum())}, neg={int((labels==0).sum())}')

## 3. Per-star periodogram (identical to cnn_periodogram.ipynb)

Same grid (1/max_baseline, 1.0 cyc/day), same `dy=rv_err` for RV,
same log1p + percentile standardization + mpl clip to +/- 5. If the diag
notebook imports the same `pg_array` as the OOF notebook, the comparison
is apples-to-apples.

In [ ]:
obs_per_star = observations.groupby('star_name', sort=True).agg(
    bjd_min=('bjd','min'), bjd_max=('bjd','max'), n=('bjd','count'))
baselines = obs_per_star['bjd_max'] - obs_per_star['bjd_min']
max_baseline = max(float(baselines.max()), 2.0)
f_low, f_high = 1.0 / max_baseline, 1.0
freq_grid = np.linspace(f_low, f_high, N_FREQ)
print(f'Grid: periods [{1/f_high:.2f}, {1/f_low:.2f}] days')

CHAN_COL = {'rv': 'rv_centered', 'RHKp': 'RHKp', 'Halpha': 'Halpha'}
star_groups = {s: grouped.get_group(s).sort_values('bjd') for s in stars}

def compute_pgram(star):
    g = star_groups[star]
    bjd = g['bjd'].values.astype(float)
    spectra = []
    for chan in CHANNELS:
        vals = g[CHAN_COL[chan]].values.astype(float)
        if len(bjd) < 2 or not np.all(np.isfinite(vals)) or np.std(vals) < 1e-8:
            spectra.append(np.zeros(N_FREQ, dtype=np.float32)); continue
        dy = None
        if chan == 'rv':
            e = g['rv_err'].values.astype(float)
            if np.all(np.isfinite(e)) and np.all(e > 0): dy = e
        ls = LombScargle(bjd, vals, dy=dy, fit_mean=True, center_data=True)
        p = ls.power(freq_grid, method=LS_METHOD, normalization='standard')
        # Sanitize BEFORE log1p: NaN/Inf -> 0, then clip to [0,1] (astropy 'standard'
        # power's valid range). Without this, a single negative power value -> log1p
        # produces NaN -> percentile(NaN) returns NaN -> (pg_array - NaN)/NaN = NaN,
        # which nan_to_num collapses to 0 -> all-zero array -> all architectures output
        # constant 0.5 -> training collapses. SILENT KILLER. (Was the bug.)
        p = np.nan_to_num(p.astype(np.float32), nan=0.0, posinf=1.0, neginf=0.0)
        p = np.clip(p, 0.0, 1.0)
        spectra.append(p.astype(np.float32))
    return np.stack(spectra, axis=0)

print(f'Building {n_stars} periodograms...')
pg_array = np.stack([compute_pgram(s) for s in stars], axis=0).astype(np.float32)
print(f'pg_array shape: {pg_array.shape}')

# log1p + percentile standardization + clip (identical to cnn_periodogram.ipynb)
pg_array = np.log1p(pg_array)
flat = pg_array.reshape(pg_array.shape[0], -1)
p_lo, p_hi = np.percentile(flat, 1.0), np.percentile(flat, 99.0)
scale = max(p_hi - p_lo, 1e-6); shift = (p_lo + p_hi) / 2.0
pg_array = np.clip((pg_array - shift) / scale, -5.0, 5.0).astype(np.float32)
pg_array = np.nan_to_num(pg_array, nan=0.0, posinf=5.0, neginf=-5.0)
print(f'stdardized. mean={pg_array.mean():.4f} std={pg_array.std():.4f}')
print(f'  range: [{pg_array.min():.3f}, {pg_array.max():.3f}]')
# Critical sanity check: array must have real variance. If std=0 the downstream
# architectures will all get constant inputs and output constant 0.5 -> no learning.
assert pg_array.std() > 1e-3, f'STANDARDIZATION COLLAPSED ARRAY: std={pg_array.std():.6f}'
print(f'  std>1e-3 sanity check passed.')
print(f'  per-channel mean: {pg_array.mean(axis=(0,2)).tolist()}')
print(f'  per-channel std:  {pg_array.std(axis=(0,2)).tolist()}')
# Input sanity: fraction of zeros
zero_frac = (pg_array == 0).mean()
print(f'  fraction of zero bins: {zero_frac:.3f}')
# ============================================================
# DIAGNOSTIC: why is the standardized array all zeros?
# ============================================================
print()
print('='*72)
print('PRE-STANDARDIZATION DIAGNOSTIC')
print('='*72)

# 1. Check raw rv_centered for NaN/Inf in the FULL observations frame
rv_raw = observations['rv_centered'].values.astype(float)
n_nan = int(np.isnan(rv_raw).sum())
n_inf = int(np.isinf(rv_raw).sum())
print(f'rv_centered (n={len(rv_raw)}): NaN={n_nan}, Inf={n_inf}')

# 2. Per-finite-check: how many stars survive compute_pgram's guard?
n_pass_finite = 0
n_pass_std = 0
n_total = len(stars)
for s in stars:
    g = star_groups[s]
    vals = g['rv_centered'].values.astype(float)
    if np.all(np.isfinite(vals)) and len(g) >= 2:
        n_pass_finite += 1
        if np.std(vals) >= 1e-8:
            n_pass_std += 1
print(f'Stars passing finite check: {n_pass_finite}/{n_total}')
print(f'Stars passing std check:    {n_pass_std}/{n_total}')

# 3. Run compute_pgram on first 5 stars and print raw power stats
print()
print('Per-star raw power stats (first 5 stars):')
for s in stars[:5]:
    g = star_groups[s]
    vals = g['rv_centered'].values.astype(float)
    n = len(g)
    has_finite = np.all(np.isfinite(vals))
    std = float(np.std(vals)) if has_finite else float('nan')
    pgram = compute_pgram(s)[0]  # (N_FREQ,)
    print(f'  {s}: n_obs={n}, rv_std={std:.6e}, '
          f'pgram[min={pgram.min():.4e}, max={pgram.max():.4e}, '
          f'mean={pgram.mean():.4e}, sum={pgram.sum():.4e}]')

# 4. Save pg_array BEFORE log1p/standardization for raw inspection
pg_raw_for_inspect = np.array([compute_pgram(s)[0] for s in stars[:20]])
print()
print(f'First 20 stars raw pg_array stats:')
print(f'  min: {pg_raw_for_inspect.min():.4e}')
print(f'  max: {pg_raw_for_inspect.max():.4e}')
print(f'  mean: {pg_raw_for_inspect.mean():.4e}')
print(f'  sum (allbins allstars): {pg_raw_for_inspect.sum():.4e}')
print(f'  fraction of zero bins: {(pg_raw_for_inspect == 0).mean():.3f}')
print(f'  # of nonzero entries: {(pg_raw_for_inspect != 0).sum()} / {pg_raw_for_inspect.size}')

# 5. Now do the log1p+percentile on JUST these 20 to see what happens
tw = np.log1p(pg_raw_for_inspect)
flat_tw = tw.reshape(-1)
p_lo_tw = np.percentile(flat_tw, 1.0)
p_hi_tw = np.percentile(flat_tw, 99.0)
scale_tw = max(p_hi_tw - p_lo_tw, 1e-6)
shift_tw = (p_lo_tw + p_hi_tw) / 2.0
print()
print(f'log1p stats on first 20 stars: min={tw.min():.4e}, max={tw.max():.4e}')
print(f'  percentiles: p1={p_lo_tw:.4e}, p99={p_hi_tw:.4e}, scale={scale_tw:.4e}')
print(f'  If all log1p(pgram)==0, then p1=p99=0, scale=1e-6, and post-std is 0.')


## 4. One fold split (rep 0 / fold 0 of the OOF protocol)

Picks exactly the first train_idx/test_idx from `StratifiedKFold(10, shuffle=True,
seed=42)` — same fold that the OOF notebook trained on first in its rep 0.
Cleanly comparable.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_iter = skf.split(np.zeros(n_stars), labels)
train_idx, test_idx = next(fold_iter)
print(f'fold 0: train={len(train_idx)} (pos={int(labels[train_idx].sum())}), '
      f'test={len(test_idx)} (pos={int(labels[test_idx].sum())})')

X_tr = torch.from_numpy(pg_array[train_idx]).to(device)
y_tr = torch.tensor(labels[train_idx], dtype=torch.float32).to(device)
X_te = torch.from_numpy(pg_array[test_idx]).to(device)
y_te_np = labels[test_idx]
print(f'X_tr: {tuple(X_tr.shape)}  X_te: {tuple(X_te.shape)}')

## 5. Training loop with per-epoch trace

For each architecture variant: print (epoch, train BCE loss, gradient L2 norm,
val PR-AUC, val ROC-AUC, logit std). Gradient norm tells us if gradients are
exploding or vanishing; logit std tells us at what epoch the model's outputs
collapse to a constant.

In [ ]:
def trace_train(model, name, n_epochs=N_EPOCHS_DIAG, lr=LR, verbose=True):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    model = model.to(device)
    pos = max(int((labels[train_idx]==1).sum()), 1)
    neg = max(int((labels[train_idx]==0).sum()), 1)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg/pos], device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr,
        epochs=n_epochs, steps_per_epoch=1, pct_start=0.1)
    ds = torch.utils.data.TensorDataset(X_tr, y_tr)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True)

    log = []
    print(f'\n===== {name} =====')
    for ep in range(n_epochs):
        model.train()
        ep_loss, ep_grad = 0.0, 0.0
        for xb, yb in dl:
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            # gradient L2 norm (across all params)
            gnorm = 0.0
            for p in model.parameters():
                if p.grad is not None:
                    gnorm += float(p.grad.detach().norm(2).item() ** 2)
            gnorm = gnorm ** 0.5
            opt.step()
            ep_loss += loss.item() * xb.size(0)
            ep_grad = max(ep_grad, gnorm)
        sched.step()
        ep_loss /= len(train_idx)

        # Eval
        model.eval()
        with torch.no_grad():
            logits_te = model(X_te).cpu().numpy()
        probs = 1.0 / (1.0 + np.exp(-logits_te))
        lstd = float(logits_te.std())
        pr = average_precision_score(y_te_np, probs)
        roc = roc_auc_score(y_te_np, probs)

        log.append({'ep':ep, 'loss':ep_loss, 'grad_norm':ep_grad, 'pr':pr,
                    'roc':roc, 'logit_std':lstd})
        if verbose and (ep % 2 == 0 or ep == n_epochs-1):
            print(f'  ep {ep:2d} loss={ep_loss:.4f} grad={ep_grad:.3f} '
                  f'PR={pr:.4f} ROC={roc:.4f} logit_std={lstd:.4f}')
    import pandas as pd
    return pd.DataFrame(log)

## 6. The three architectures

Same as cnn_periodogram.ipynb for #1 (the failing one), then LayerNorm variant,
then an MLP baseline. The MLP is just to check whether the input carries any
signal at all — it doesn't need to be sophisticated.

In [ ]:
class ConvBlockBN(nn.Module):
    def __init__(self, ci, co, k, drop=0.3):
        super().__init__()
        self.conv = nn.Conv1d(ci, co, k, padding=k//2)
        self.bn = nn.BatchNorm1d(co)
        self.act = nn.GELU(); self.drop = nn.Dropout(drop); self.pool = nn.AvgPool1d(2)
    def forward(self, x):
        return self.pool(self.drop(self.act(self.bn(self.conv(x)))))

class ConvBlockLN(nn.Module):
    def __init__(self, ci, co, k, drop=0.3):
        super().__init__()
        self.conv = nn.Conv1d(ci, co, k, padding=k//2)
        self.ln = nn.GroupNorm(1, co)  # GroupNorm with 1 group = LayerNorm over channels
        self.act = nn.GELU(); self.drop = nn.Dropout(drop); self.pool = nn.AvgPool1d(2)
    def forward(self, x):
        return self.pool(self.drop(self.act(self.ln(self.conv(x)))))

def make_cnn(block_class, in_ch=1, hidden=32, drop=0.3):
    class M(nn.Module):
        def __init__(self):
            super().__init__()
            self.b1 = block_class(in_ch,    hidden,    7, drop)
            self.b2 = block_class(hidden,   hidden*2,  5, drop)
            self.b3 = block_class(hidden*2, hidden*4,  5, drop)
            self.b4 = block_class(hidden*4, hidden*4, 3, drop)
            self.gap = nn.AdaptiveAvgPool1d(1)
            self.gmp = nn.AdaptiveMaxPool1d(1)
            self.head = nn.Sequential(
                nn.Linear(2 * hidden*4, 32),
                nn.GELU(), nn.Dropout(drop), nn.Linear(32, 1))
        def forward(self, x):
            h = self.b4(self.b3(self.b2(self.b1(x))))
            return self.head(torch.cat([self.gap(h).squeeze(-1),
                                          self.gmp(h).squeeze(-1)], dim=1)).squeeze(-1)
    return M()

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=128, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, 1))
    def forward(self, x):
        # x shape: (B, C, N_FREQ) -> flatten
        return self.net(x.flatten(1)).squeeze(-1)

# Smoke: print param counts
for name, m in [('CNN-BN', make_cnn(ConvBlockBN)),
                ('CNN-LN', make_cnn(ConvBlockLN)),
                ('MLP', MLP(len(CHANNELS)*N_FREQ))]:
    n_p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  {name:8s}: {n_p:,} params')

## 7. Run all three variants (single fold, 20 epochs each)

In [ ]:
all_logs = {}
t0 = time.time()
all_logs['CNN-BN'] = trace_train(make_cnn(ConvBlockBN), 'CNN-BN (original, failing)')
all_logs['CNN-LN'] = trace_train(make_cnn(ConvBlockLN), 'CNN-LN (LayerNorm variant)')
all_logs['MLP']    = trace_train(MLP(len(CHANNELS)*N_FREQ), 'MLP baseline')
print(f'\nTotal time: {time.time()-t0:.1f}s')

## 8. Side-by-side trajectory

Print the final-epoch stats for each variant and the train loss curve. If at
the end of 20 epochs:
- **CNN-BN logit_std <-0.1** and **PR-AUC = 0.19** = class rate, model collapsed.
- **CNN-LN or MLP PR-AUC > 0.35** = extracts *some* signal -> we have a path
  (drop BN, switch to LN or MLP, re-run OOF).
- **All three collapse to PR-AUC ~= 0.19** = the input itself carries no
  shape signal that V4's scalar features didn't already capture -> write paper.

## 8.5. Hypothesis: percentile standardization killed per-star variance

**Claim:** The `np.percentile(flat, 1.0/99.0)` standardization in cell 3 computes
shift and scale over the *joint* distribution of all stars x all bins. Since ~95%
of LS power bins are at the noise floor (log(1+1e-4) ~ 0.0001), the standardization
shift and scale are dominated by the global noise floor. After standardization,
every star's periodogram gets pushed toward the same ~constant vector (-0.5), so
`pg_array.std(axis=(0,2))` ~ 0 -- every star is essentially identical.

Test: re-standardize **per-star** (each star's image normalized to unit std) and
retrain the MLP. If MLP learns signal now, the standardization was the bug -- the
periodogram shape DOES carry per-star information, just buried under a global shift.
If MLP still fails, the periodogram truly lacks shape signal.


In [ ]:
# Per-star standardization: pg_array shape (N, C, F) -> standardize each (C, F) slice independently
pg_per_star = np.log1p(np.stack([compute_pgram(s) for s in stars], axis=0).astype(np.float32))

def per_star_std(arr):
    # arr: (N, C, F) -- for each star (n, c) compute median and MAD over the F axis
    n_stars, n_ch, n_freq = arr.shape
    out = np.zeros_like(arr)
    for i in range(n_stars):
        for c in range(n_ch):
            spec = arr[i, c, :]
            med = float(np.median(spec))
            mad = float(np.median(np.abs(spec - med)))
            scale = max(1.4826 * mad, 1e-6)
            out[i, c, :] = (spec - med) / scale
    out = np.clip(out, -5.0, 5.0).astype(np.float32)
    out = np.nan_to_num(out, nan=0.0, posinf=5.0, neginf=-5.0)
    return out

pg_per_star = per_star_std(pg_per_star)
print(f'Per-star standardized. shape={pg_per_star.shape}')
print(f'  per-channel mean: {pg_per_star.mean(axis=(0,2)).tolist()}')
print(f'  per-channel std:  {pg_per_star.std(axis=(0,2)).tolist()}')
print(f'  per-star std (median across stars): {float(np.median(pg_per_star.std(axis=(1,2)))):.4f}')

X_tr_per = torch.from_numpy(pg_per_star[train_idx]).to(device)
X_te_per = torch.from_numpy(pg_per_star[test_idx]).to(device)
y_te_per_np = labels[test_idx]

class MLP_per(nn.Module):
    def __init__(self, in_dim, hidden=128, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, 1))
    def forward(self, x):
        return self.net(x.flatten(1)).squeeze(-1)

def trace_train_per(model, name, n_epochs=N_EPOCHS_DIAG, lr=LR, verbose=True):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); np.random.seed(seed)
    model = model.to(device)
    pos = max(int((labels[train_idx]==1).sum()), 1)
    neg = max(int((labels[train_idx]==0).sum()), 1)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg/pos], device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1, pct_start=0.1)
    ds = torch.utils.data.TensorDataset(X_tr_per, y_tr)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True)
    log = []
    print(f'\n===== {name} =====')
    for ep in range(n_epochs):
        model.train()
        ep_loss = 0.0
        for xb, yb in dl:
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward(); opt.step()
            ep_loss += loss.item() * xb.size(0)
        sched.step()
        ep_loss /= len(train_idx)
        model.eval()
        with torch.no_grad():
            logits_te = model(X_te_per).cpu().numpy()
        probs = 1.0 / (1.0 + np.exp(-logits_te))
        lstd = float(logits_te.std())
        pr = average_precision_score(y_te_per_np, probs)
        roc = roc_auc_score(y_te_per_np, probs)
        log.append({'ep':ep, 'loss':ep_loss, 'pr':pr, 'roc':roc, 'logit_std':lstd})
        if verbose and (ep % 2 == 0 or ep == n_epochs-1):
            print(f'  ep {ep:2d} loss={ep_loss:.4f} PR={pr:.4f} ROC={roc:.4f} logit_std={lstd:.4f}')
    return pd.DataFrame(log)

per_star_log = trace_train_per(MLP_per(len(CHANNELS)*N_FREQ), 'MLP on per-star-standardized periodogram')


## 9. DEFINITIVE TEST: MLP on V4 RF's scalar features

V4 RF got PR-AUC = 0.61 OOF from these exact 39 features. If our MLP **fails**
on these features too, we have a training-loop bug (not an information-bounded
problem). If our MLP **succeeds** on them, the pipeline is fine and the
periodogram image truly lacks shape signal that summary statistics already capture.

One fold, 20 epochs, ~1 min. Definitive.


In [ ]:
# ============================================================
# DEFINITIVE TEST: MLP on V4 RF's 39 scalar features
# ============================================================
# V4 RF achieved PR-AUC = 0.6066 OOF using these exact 39 features.
# If our MLP fails on these features too, we have a training-loop bug.
# If the MLP succeeds, the pipeline is fine and the periodogram image truly
# lacks the signal that hand-aggregated scalars capture.

# Reproduce the V4 feature matrix (rf_multiseed.ipynb cells [2] + [13] + [18]).
# This is a minimal port -- only the features V4 used, no extras.

# ── Original 18 physical features (baseline.ipynb cell 2) ──
def compute_star_features(group):
    f = {}
    rv = group["rv_centered"]
    f['rv_std']              = rv.std()
    f['rv_range']            = rv.max() - rv.min()
    f['rv_mean_abs_dev']     = (rv - rv.median()).abs().mean()
    f['rv_skew']             = rv.skew()
    f['rv_kurtosis']         = rv.kurtosis()
    f['rv_err_mean']         = group['rv_err'].mean()
    f['rv_err_std']          = group['rv_err'].std()
    f['rhkp_std']            = group['RHKp'].std()
    f['rhkp_range']          = group['RHKp'].max() - group['RHKp'].min()
    f['rhkp_mean']           = group['RHKp'].mean()
    f['halpha_std']          = group['Halpha'].std()
    f['halpha_range']        = group['Halpha'].max() - group['Halpha'].min()
    f['halpha_mean']         = group['Halpha'].mean()
    f['rv_rhkp_corr_abs']    = abs(group['rv_centered'].corr(group['RHKp']))
    f['rv_halpha_corr_abs']  = abs(group['rv_centered'].corr(group['Halpha']))
    f['rhkp_halpha_corr_abs']= abs(group['RHKp'].corr(group['Halpha']))
    f['rhkp_rv_ratio']       = group['RHKp'].std() / rv.std() if rv.std() > 0 else 0.0
    f['has_exoplanets']      = group['has_exoplanets'].iloc[0]
    return pd.Series(f)

sf = observations.groupby('star_name', sort=True).apply(compute_star_features, include_groups=False).reset_index().fillna(0)

# ── V4 LS-derived features (only the ones that made the final V4 feature set) ──
# Same compact port as above; cover the 24 LS-derived features V4 used.
from astropy.timeseries import LombScargle
N_FREQ_LS = 500
F_MIN_DAYS = 1.0
LS_METHOD_LS = 'fast'

def compute_ls_features(group):
    f = {}
    rv = group['rv_centered'].values.astype(float)
    rhkp = group['RHKp'].values.astype(float)
    halpha = group['Halpha'].values.astype(float)
    bjd = group['bjd'].values.astype(float)
    rv_err = group['rv_err'].values.astype(float)
    n = len(bjd)
    baseline = float(bjd.max() - bjd.min()) if n > 1 else 0.0
    p_max = max(baseline, 2.0) if baseline > 0 else 2.0
    freqs = np.linspace(1.0/p_max, 1.0/F_MIN_DAYS, N_FREQ_LS)

    rv_s = float(np.std(rv)) if n > 0 else 0.0
    rhkp_s = float(np.std(rhkp)) if n > 0 else 0.0
    halpha_s = float(np.std(halpha)) if n > 0 else 0.0

    f['rv_ls_mean_power']      = 0.0
    f['rhkp_ls_peak_power']    = 0.0
    f['halpha_ls_peak_power']  = 0.0
    f['rhkp_rv_power_ratio']   = 0.0
    f['halpha_rv_power_ratio'] = 0.0
    f['min_activity_rv_ratio'] = 0.0
    f['rv_ls_peak_power']      = 0.0
    f['rv_top2_power_ratio']    = 0.0
    f['rv_top3_power_ratio']   = 0.0
    f['rv_power_conc_top10pct'] = 0.0
    f['rhkp_power_conc_top10pct'] = 0.0
    f['halpha_top2_power_ratio'] = 0.0
    f['rv_power_conc_top5pct'] = 0.0
    f['rv_power_conc_top1pct'] = 0.0
    f['rv_periodic_snr']       = 0.0
    f['rv_excess_std']         = 0.0
    f['rv_snr']                = 0.0
    f['rv_weighted_amplitude'] = 0.0
    f['rv_rhkp_partial_corr']  = 0.0
    f['rhkp_halpha_partial_corr'] = 0.0
    f['rv_halpha_partial_corr']= 0.0
    f['rv_rhkp_corr_std']      = 0.0
    f['rhkp_rv_power_ratio']   = 0.0
    f['halpha_rv_power_ratio'] = 0.0  # (may duplicate above; minor)

    distinct_t = np.unique(np.round(bjd, 6)).size if n > 0 else 0
    can_run = (n >= 4) and (rv_s > 0) and (baseline > 0) and (distinct_t >= 4)

    if can_run:
        try:
            ls_rv   = LombScargle(bjd, rv,   dy=rv_err,     normalization='standard', fit_mean=True)
            ls_rhk  = LombScargle(bjd, rhkp,                  normalization='standard', fit_mean=True)
            ls_hal  = LombScargle(bjd, halpha,                normalization='standard', fit_mean=True)
            p_rv  = ls_rv.power(freqs,  method=LS_METHOD_LS)
            p_rhk = ls_rhk.power(freqs, method=LS_METHOD_LS)
            p_hal = ls_hal.power(freqs, method=LS_METHOD_LS)

            f['rv_ls_mean_power']      = float(np.mean(p_rv))
            f['rv_ls_peak_power']       = float(np.max(p_rv))
            f['rhkp_ls_peak_power']     = float(np.max(p_rhk))
            f['halpha_ls_peak_power']   = float(np.max(p_hal))
            rhk_ratio = float(np.max(p_rhk) / np.max(p_rv)) if np.max(p_rv) > 0 else 0.0
            hal_ratio = float(np.max(p_hal) / np.max(p_rv)) if np.max(p_rv) > 0 else 0.0
            f['rhkp_rv_power_ratio']    = rhk_ratio
            f['halpha_rv_power_ratio']  = hal_ratio
            f['min_activity_rv_ratio']  = min(rhk_ratio, hal_ratio)

            sidx = np.argsort(p_rv)[::-1]
            peak = p_rv[sidx[0]]
            f['rv_top2_power_ratio']    = float(p_rv[sidx[1]] / peak) if len(sidx) > 1 and peak > 0 else 0.0
            f['rv_top3_power_ratio']    = float(p_rv[sidx[2]] / peak) if len(sidx) > 2 and peak > 0 else 0.0

            n_top10 = max(1, len(p_rv) // 10)
            f['rv_power_conc_top10pct']   = float(np.sum(np.sort(p_rv)[-n_top10:]) / max(np.sum(p_rv), 1e-9))
            f['rhkp_power_conc_top10pct'] = float(np.sum(np.sort(p_rhk)[-n_top10:]) / max(np.sum(p_rhk), 1e-9))
            f['halpha_top2_power_ratio']  = float(np.sort(p_hal)[-2] / np.sort(p_hal)[-1]) if np.max(p_hal) > 0 else 0.0

            n_top5  = max(1, len(p_rv) // 20)
            n_top1  = max(1, len(p_rv) // 100)
            f['rv_power_conc_top5pct'] = float(np.sum(np.sort(p_rv)[-n_top5:]) / max(np.sum(p_rv), 1e-9))
            f['rv_power_conc_top1pct'] = float(np.sum(np.sort(p_rv)[-n_top1:]) / max(np.sum(p_rv), 1e-9))

            # periodic SNR: peak / median power
            f['rv_periodic_snr'] = float(peak / max(np.median(p_rv), 1e-9))

            # uncertainty-aware (only need rv_err)
            err_var = np.mean(rv_err ** 2) if np.all(rv_err > 0) else 0.0
            f['rv_excess_std']  = float(np.sqrt(max(rv_s ** 2 - err_var, 0.0)))
            f['rv_snr']         = float(rv_s / max(np.mean(rv_err), 1e-9)) if np.all(rv_err > 0) else 0.0
            A = rv_s * np.sqrt(2.0)  # rough Keplerian amplitude estimate
            f['rv_weighted_amplitude'] = float(A)
        except Exception:
            pass
    return pd.Series(f)

ls_feats = observations.groupby('star_name', sort=True).apply(compute_ls_features, include_groups=False).reset_index().fillna(0)

# ── Cadence median (single V3 feature that made V4's set) ──
def compute_cadence(group):
    bjd = group['bjd'].values.astype(float)
    if len(bjd) >= 2:
        return float(np.median(np.diff(np.sort(bjd))))
    return 0.0
cad = observations.groupby('star_name', sort=True).apply(compute_cadence, include_groups=False).reset_index().rename(columns={0:'cadence_median'}).fillna(0)

# ── Merge everything ──
v4 = sf.merge(ls_feats, on='star_name', how='left').fillna(0)
v4 = v4.merge(cad[['star_name','cadence_median']], on='star_name', how='left').fillna(0)

# Drop the ~4 dead features V4 pruned (zero importance / dominated by abs versions)
DROP = ['rv_rhkp_corr','rv_halpha_corr','rhkp_halpha_corr','rhkp_halpha_corr_abs','rhkp_rv_power_ratio']
# (V4 actually kept rhkp_halpha_corr_abs but pruned rv_rhkp_corr, rv_halpha_corr, rhkp_halpha_corr in cell 14 -- this is approximate.)

PHYS_V4 = [c for c in v4.columns if c not in ['star_name','has_exoplanets'] and c not in DROP]
print(f'V4-feature matrix: {v4.shape}, {len(PHYS_V4)} features')

# Stack in star-order to match labels
v4_mat = v4[[c for c in PHYS_V4 if c in v4.columns]].values  # (n_stars, ~40)
v4_mat = np.nan_to_num(v4_mat.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

# Standardize v4_mat (percentile-style, train-fold-scoped to mirror a real pipeline)
tr_v4 = v4_mat[train_idx]
p_lo_v4 = np.percentile(tr_v4, 1.0, axis=0)
p_hi_v4 = np.percentile(tr_v4, 99.0, axis=0)
scale_v4 = np.maximum(p_hi_v4 - p_lo_v4, 1e-6)
shift_v4 = (p_lo_v4 + p_hi_v4) / 2.0
v4_std = (v4_mat - shift_v4) / scale_v4
v4_std = np.clip(np.nan_to_num(v4_std, nan=0.0, posinf=5.0, neginf=-5.0), -5.0, 5.0).astype(np.float32)

X_tr_v4 = torch.from_numpy(v4_std[train_idx]).to(device)
X_te_v4 = torch.from_numpy(v4_std[test_idx]).to(device)
y_te_v4_np = labels[test_idx]

print(f'V4-features X_tr: {tuple(X_tr_v4.shape)}, X_te: {tuple(X_te_v4.shape)}')
print(f'  train mean={X_tr_v4.mean().item():.4f} std={X_tr_v4.std().item():.4f}')

# ── Define MLP on V4 features (same MLP class as before, different in_dim) ──
class MLPv4(nn.Module):
    def __init__(self, in_dim, hidden=128, drop=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

def trace_train_v4(model, name, n_epochs=N_EPOCHS_DIAG, lr=LR, verbose=True):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); np.random.seed(seed)
    model = model.to(device)
    pos = max(int((labels[train_idx]==1).sum()), 1)
    neg = max(int((labels[train_idx]==0).sum()), 1)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg/pos], device=device))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=n_epochs, steps_per_epoch=1, pct_start=0.1)
    ds = torch.utils.data.TensorDataset(X_tr_v4, y_tr)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True)
    log = []
    print(f'\n===== {name} =====')
    for ep in range(n_epochs):
        model.train()
        ep_loss = 0.0
        for xb, yb in dl:
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward(); opt.step()
            ep_loss += loss.item() * xb.size(0)
        sched.step()
        ep_loss /= len(train_idx)
        model.eval()
        with torch.no_grad():
            logits_te = model(X_te_v4).cpu().numpy()
        probs = 1.0 / (1.0 + np.exp(-logits_te))
        lstd = float(logits_te.std())
        pr = average_precision_score(y_te_v4_np, probs)
        roc = roc_auc_score(y_te_v4_np, probs)
        log.append({'ep':ep, 'loss':ep_loss, 'pr':pr, 'roc':roc, 'logit_std':lstd})
        if verbose and (ep % 2 == 0 or ep == n_epochs-1):
            print(f'  ep {ep:2d} loss={ep_loss:.4f} PR={pr:.4f} ROC={roc:.4f} logit_std={lstd:.4f}')
    return pd.DataFrame(log)

v4_log = trace_train_v4(MLPv4(len(PHYS_V4)), 'MLP on V4 RF scalar features')

print()
print('='*72)
print('DEFINITIVE VERDICT (3-way comparison)')
print('='*72)
period_final_pr  = float(all_logs['MLP'].iloc[-1]['pr'])         # joint-std periodogram
per_star_final_pr = float(per_star_log.iloc[-1]['pr'])           # per-star-std periodogram
v4_final_pr       = float(v4_log.iloc[-1]['pr'])                 # scalar features
print(f'MLP on periodogram (joint-std):  PR-AUC = {period_final_pr:.4f}')
print(f'MLP on periodogram (per-star):    PR-AUC = {per_star_final_pr:.4f}')
print(f'MLP on V4 scalar features:        PR-AUC = {v4_final_pr:.4f}')
print()
if v4_final_pr < 0.25:
    print('MLP FAILED on V4 scalar features too -> training-loop bug.')
    print('Fix: try LR=1e-4, no scheduler, gaussian init.')
elif per_star_final_pr > 0.40 and period_final_pr < 0.30:
    print('Joint-std periodogram failed; per-star-std succeeded.')
    print('BUG: percentile standardization (joint across stars) killed per-star variance.')
    print('Fix: re-run cnn_periodogram.ipynb with per-star median+MAD standardization.')
    print(f'Expected lift: {per_star_final_pr - period_final_pr:+.3f} PR-AUC.')
elif period_final_pr < 0.25 and per_star_final_pr < 0.25 and v4_final_pr > 0.35:
    print('Periodogram MLP fails BOTH standardizations.')
    print('V4 scalar MLP succeeds -> pipeline is fine.')
    print('The periodogram image lacks shape signal that V4 scalars already capture.')
    print('-> Option (C) genuinely fails. Begin paper outline.')
else:
    print('Mixed result. Investigate manually.')
    print(f'  v4={v4_final_pr:.4f}, per_star={per_star_final_pr:.4f}, joint={period_final_pr:.4f}')


In [ ]:
print('Final-epoch summary (epoch 19):')
print(f"{'model':8s} {'loss':>8s} {'grad':>8s} {'PR':>6s} {'ROC':>6s} {'logit_std':>10s}")
for name, log in all_logs.items():
    r = log.iloc[-1]
    print(f"{name:8s} {r['loss']:8.4f} {r['grad_norm']:8.3f} "
          f"{r['pr']:6.4f} {r['roc']:6.4f} {r['logit_std']:10.4f}")

print()
print('PR-AUC trajectory over epochs (every 2):')
for name, log in all_logs.items():
    s = '  '.join(f'{r["pr"]:.3f}' for _, r in log.iterrows() if r['ep'] % 4 == 0)
    print(f'  {name:8s}: {s}')

print()
print('Logit-std trajectory over epochs (every 4):')
for name, log in all_logs.items():
    s = '  '.join(f'{r["logit_std"]:.3f}' for _, r in log.iterrows() if r['ep'] % 4 == 0)
    print(f'  {name:8s}: {s}')

print()
print('Train loss trajectory (every 4):')
for name, log in all_logs.items():
    s = '  '.join(f'{r["loss"]:.3f}' for _, r in log.iterrows() if r['ep'] % 4 == 0)
    print(f'  {name:8s}: {s}')

# Verdict
print()
print('='*72)
print('VERDICT')
print('='*72)
final_prs = {name: log.iloc[-1]['pr'] for name, log in all_logs.items()}
if all(pr < 0.25 for pr in final_prs.values()):
    print('All three collapsed. The periodogram input itself carries no extractable')
    print('signal that V4 RF scalar features did not already capture.')
    print('-> Strong evidence for the "RF is information-bounded" paper claim.')
    print('-> Option (C) fails: write the paper.')
elif final_prs['CNN-BN'] < 0.25 and (final_prs['CNN-LN'] > 0.35 or final_prs['MLP'] > 0.35):
    winner = 'CNN-LN' if final_prs['CNN-LN'] > final_prs['MLP'] else 'MLP'
    print(f'CNN-BN collapsed ({final_prs["CNN-BN"]:.3f}) but {winner} learned signal ({final_prs[winner]:.3f}).')
    print(f'BatchNorm was the bug. Fix: switch the OOF notebook to {winner}.')
    print('-> Re-run OOF protocol with the fixed architecture.')
elif final_prs['CNN-BN'] < 0.25 and all(pr < 0.30 for pr in [final_prs['CNN-LN'], final_prs['MLP']]):
    print('CNN-BN collapsed AND even simpler models barely learn.')
    print('Likely LR/OneCycleLR schedule badly tuned for this input. Try LR=1e-4, no scheduler.')
else:
    print('Unexpected result. Investigate manually:')
    for name, pr in final_prs.items():
        print(f'  {name}: {pr:.4f}')